## Introduction 

Cette leçon couvrira : 
- Qu'est-ce que l'appel de fonction et ses cas d'utilisation 
- Comment créer un appel de fonction avec Azure OpenAI 
- Comment intégrer un appel de fonction dans une application 

## Objectifs d'apprentissage 

Après avoir terminé cette leçon, vous saurez comment et comprendrez : 

- Le but de l'utilisation de l'appel de fonction 
- Configurer l'appel de fonction en utilisant le service Azure Open AI 
- Concevoir des appels de fonctions efficaces pour le cas d'utilisation de votre application 


## Comprendre les appels de fonctions 

Pour cette leçon, nous voulons créer une fonctionnalité pour notre startup éducative qui permet aux utilisateurs d'utiliser un chatbot pour trouver des cours techniques. Nous recommanderons des cours adaptés à leur niveau de compétence, leur poste actuel et la technologie qui les intéresse. 

Pour cela, nous utiliserons une combinaison de : 
 - `Azure Open AI` pour créer une expérience de chat pour l'utilisateur
 - `Microsoft Learn Catalog API` pour aider les utilisateurs à trouver des cours en fonction de leur demande 
 - `Function Calling` pour prendre la requête de l'utilisateur et l'envoyer à une fonction afin de faire la demande API. 

Pour commencer, examinons pourquoi nous voudrions utiliser les appels de fonction en premier lieu : 


### Pourquoi l'appel de fonction 

Si vous avez suivi un autre cours de cette formation, vous comprenez probablement la puissance de l'utilisation des grands modèles de langage (LLM). Espérons que vous pouvez aussi voir certaines de leurs limites. 

L'appel de fonction est une fonctionnalité du service Azure Open AI pour surmonter les limitations suivantes : 
1) Format de réponse cohérent 
2) Capacité à utiliser des données provenant d'autres sources d'une application dans un contexte de chat 

Avant l'appel de fonction, les réponses d'un LLM étaient non structurées et incohérentes. Les développeurs devaient écrire un code de validation complexe pour pouvoir gérer chaque variation d'une réponse. 

Les utilisateurs ne pouvaient pas obtenir de réponses comme "Quel est le temps actuel à Stockholm ?". Cela est dû au fait que les modèles étaient limités au moment où les données ont été entraînées. 

Voyons l'exemple ci-dessous qui illustre ce problème : 

Supposons que nous voulons créer une base de données des données des étudiants afin de leur suggérer le bon cours. Ci-dessous, nous avons deux descriptions d'étudiants qui sont très similaires dans les données qu'ils contiennent.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Nous voulons envoyer ceci à un LLM pour analyser les données. Cela peut ensuite être utilisé dans notre application pour l'envoyer à une API ou le stocker dans une base de données. 

Créons deux invites identiques pour indiquer au LLM quelles informations nous intéressent : 


Nous voulons envoyer cela à un LLM pour analyser les parties importantes pour notre produit. Ainsi, nous pouvons créer deux invites identiques pour instruire le LLM : 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Après avoir créé ces deux prompts, nous les enverrons au LLM en utilisant `client.responses.create`. Nous stockons le prompt dans la variable `input` et attribuons le rôle `user`. Cela permet d'imiter un message écrit par un utilisateur à un chatbot. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Nous pouvons maintenant envoyer les deux requêtes au LLM et examiner la réponse que nous recevons. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Même si les invites sont les mêmes et que les descriptions sont similaires, nous pouvons obtenir différents formats pour la propriété `Grades`.

Si vous exécutez la cellule ci-dessus plusieurs fois, le format peut être `3.7` ou `3.7 GPA`.

Cela s'explique par le fait que le LLM prend des données non structurées sous la forme de l'invite écrite et retourne également des données non structurées. Nous devons avoir un format structuré afin de savoir à quoi nous attendre lors du stockage ou de l'utilisation de ces données.

En utilisant l'appel fonctionnel, nous pouvons nous assurer de recevoir des données structurées en retour. Lors de l'utilisation de l'appel fonctionnel, le LLM n'appelle ni n'exécute réellement aucune fonction. Au lieu de cela, nous créons une structure que le LLM doit suivre pour ses réponses. Nous utilisons ensuite ces réponses structurées pour savoir quelle fonction exécuter dans nos applications.
 


![Diagramme du flux d’appel de fonction](../../../../translated_images/fr/Function-Flow.083875364af4f4bb.webp)


Nous pouvons ensuite prendre ce qui est retourné par la fonction et le renvoyer au LLM. Le LLM répondra alors en utilisant un langage naturel pour répondre à la requête de l'utilisateur. 


### Cas d'utilisation des appels de fonction 

**Appeler des outils externes** 
Les chatbots sont excellents pour fournir des réponses aux questions des utilisateurs. En utilisant les appels de fonction, les chatbots peuvent utiliser les messages des utilisateurs pour accomplir certaines tâches. Par exemple, un étudiant peut demander au chatbot « Envoyer un email à mon professeur pour dire que j'ai besoin de plus d'aide sur ce sujet ». Cela peut faire un appel de fonction à `send_email(to: string, body: string)`


**Créer des requêtes API ou base de données**
Les utilisateurs peuvent trouver des informations en utilisant un langage naturel qui est converti en une requête formatée ou une demande API. Un exemple pourrait être un enseignant qui demande « Qui sont les étudiants qui ont terminé le dernier devoir » ce qui pourrait appeler une fonction nommée `get_completed(student_name: string, assignment: int, current_status: string)`


**Créer des données structurées**
Les utilisateurs peuvent prendre un bloc de texte ou un CSV et utiliser le LLM pour en extraire les informations importantes. Par exemple, un étudiant peut convertir un article Wikipédia sur les accords de paix pour créer des fiches mémoires d'IA. Cela peut être fait en utilisant une fonction appelée `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Création de votre premier appel de fonction 

Le processus de création d'un appel de fonction comprend 3 étapes principales : 
1. Appeler l'API Chat Completions avec une liste de vos fonctions et un message utilisateur 
2. Lire la réponse du modèle pour effectuer une action, c’est-à-dire exécuter une fonction ou un appel API 
3. Faire un autre appel à l'API Chat Completions avec la réponse de votre fonction pour utiliser cette information afin de créer une réponse pour l'utilisateur. 


![Flux d'un appel de fonction](../../../../translated_images/fr/LLM-Flow.3285ed8caf4796d7.webp)


### Éléments d’un appel de fonction 

#### Entrée des utilisateurs 

La première étape consiste à créer un message utilisateur. Cela peut être attribué dynamiquement en prenant la valeur d'une saisie texte ou vous pouvez attribuer une valeur ici. Si c’est la première fois que vous travaillez avec l’API de Chat Completions, nous devons définir le `role` et le `content` du message. 

Le `role` peut être soit `system` (création des règles), `assistant` (le modèle) ou `user` (l’utilisateur final). Pour l’appel de fonction, nous allons attribuer ceci à `user` avec un exemple de question. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Création de fonctions. 

Ensuite, nous définirons une fonction et les paramètres de cette fonction. Nous utiliserons une seule fonction ici appelée `search_courses`, mais vous pouvez créer plusieurs fonctions.

**Important** : Les fonctions sont incluses dans le message système envoyé au LLM et seront comptabilisées dans le nombre de jetons disponibles dont vous disposez. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Définitions** 

`name` - Le nom de la fonction que nous voulons voir appelée. 

`description` - C'est la description de la façon dont la fonction fonctionne. Ici, il est important d'être précis et clair 

`parameters` - Une liste des valeurs et du format que vous souhaitez que le modèle produise dans sa réponse 


`type` - Le type de données dans lequel les propriétés seront stockées 

`properties` - Liste des valeurs spécifiques que le modèle utilisera pour sa réponse 


`name` - le nom de la propriété que le modèle utilisera dans sa réponse formatée 

`type` - Le type de données de cette propriété 

`description` - Description de la propriété spécifique 


**Optionnel**

`required` - propriété requise pour que l'appel de la fonction soit complété 


### Effectuer l'appel de fonction 
Après avoir défini une fonction, nous devons maintenant l'inclure dans l'appel à l'API Chat Completion. Nous le faisons en ajoutant `functions` à la requête. Dans ce cas, `functions=functions`. 

Il existe également une option pour définir `function_call` sur `auto`. Cela signifie que nous laisserons le LLM décider quelle fonction doit être appelée en fonction du message de l'utilisateur, plutôt que de l'assigner nous-mêmes.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Regardons maintenant la réponse et voyons comment elle est formatée : 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Vous pouvez voir que le nom de la fonction est appelé et qu'à partir du message de l'utilisateur, le LLM a pu trouver les données correspondant aux arguments de la fonction. 


## 3.Intégration des appels de fonction dans une application. 


Après avoir testé la réponse formatée du LLM, nous pouvons maintenant l'intégrer dans une application. 

### Gestion du flux 

Pour intégrer cela dans notre application, suivons les étapes suivantes : 

Tout d'abord, effectuons l'appel aux services Open AI et stockons le message dans une variable appelée `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Nous allons maintenant définir la fonction qui appellera l'API Microsoft Learn pour obtenir une liste de cours : 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Par bonne pratique, nous verrons ensuite si le modèle souhaite appeler une fonction. Après cela, nous créerons l'une des fonctions disponibles et la ferons correspondre à la fonction qui est appelée. 
Nous prendrons ensuite les arguments de la fonction et les mapperons aux arguments provenant du LLM.

Enfin, nous ajouterons le message d'appel de fonction et les valeurs qui ont été retournées par le message `search_courses`. Cela fournit au LLM toutes les informations dont il a besoin pour
répondre à l'utilisateur en utilisant un langage naturel.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Nous allons maintenant envoyer le message mis à jour au LLM afin de recevoir une réponse en langage naturel au lieu d'une réponse formatée en JSON par l'API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Défi de Code 

Excellent travail ! Pour continuer votre apprentissage de l’appel de fonctions Azure Open AI, vous pouvez créer : https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Plus de paramètres de la fonction qui pourraient aider les apprenants à trouver plus de cours. Vous pouvez trouver les paramètres d’API disponibles ici : 
 - Créer un autre appel de fonction qui prend plus d’informations de l’apprenant comme leur langue maternelle 
 - Créer une gestion des erreurs lorsque l’appel de fonction et/ou l’appel API ne retourne pas de cours adaptés 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
